# Alpha=26 deconvolution profiles

Now that the CLOCCS parameters are correct and the two replicates are producing concordante deconvolution results. We now look at the daughter specific genes, DSE1-4 to determine the alpha value that is required, per Xin:

"We calibrated the duration of the attachment period to be the smallest duration such that the decon- volved transcription profiles of all four genes are primarily within DG1"


In [15]:
# Preamble, notebook setup and imports

%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [64]:
from src.model import Model

In [60]:
def compute_prop_in_dg1(model1):
    config = model1.config
    
    dg1_indices = config.get_timepoints_phases_Hpositions_for_branch('b')[0][2]
    postg1_indices = config.get_timepoints_phases_Hpositions_for_branch('b')[1][2]

    cg1_indices = config.get_timepoints_phases_Hpositions_for_branch('t')[0][2]
    rg1_indices = config.get_timepoints_phases_Hpositions_for_branch('i')[0][2]

    f = model1.f

    dg1_f = f[dg1_indices]
    cg1_f = f[cg1_indices]
    rg1_f = f[rg1_indices]
    postg1_f = f[postg1_indices]

    return dg1_f.sum() / f.sum()

In [71]:
from src.dynamic_config_alpha import create_dynamic_alpha_config

def search_alphas(gene_name, posteriors_filepath, alphas):

    dg1_props = []

    for alpha in alphas:

        config = create_dynamic_alpha_config(posteriors_filepath, alpha, 1, "Dynamic config")

        model1 = Model(config, gene_name, 0.0)
        model1.deconvolve_find_optimal_gamma()

        dg1_prop = compute_prop_in_dg1(model1)
        dg1_props.append(dg1_prop)
        
    ret_df = pd.DataFrame({"alpha": alphas, "prop_dg1": dg1_props})
    return ret_df

In [ ]:

from src.timer import Timer

timer = Timer()

alphas = np.arange(0, 40, 10)

posteriors_filepath = 'data/yl_2019_replicate1/posteriors.txt'
gene_name = "DSE2"

timer = Timer()
print(f"Computing {len(alphas)} alpha values for {gene_name}...", end="")
dse2_dg1_df = search_alphas(gene_name, posteriors_filepath, alphas)
print(f"Done in {timer.get_time()}")


Computing 4 alpha values for DSE2...

In [ ]:
dse2_dg1_df